In [1]:
#!/usr/bin/env python

import os
import json
import joblib
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# CONFIG
model_dir = Path("./Ridge_OTU_Trained_Models/")
ukb_metadata_path = "variable_mapping/ukb_as_agp_metadata.csv"
ukb_index_col = "sample_name"

# outcome variable in UKB-like file
cognition_col = "fluid_intelligence_score"

# covariates (match your existing projection style)
cov_num = ["age_corrected"]                 # continuous
cov_cat = ["race", "country_of_birth"]      # categorical

out_dir = Path("./hongrui_result/Ridge_CV_Results/UKB_Ridge_OTU_Projection/")
out_dir.mkdir(parents=True, exist_ok=True)

# filter rows
# - We skip some high-missing lifestyle columns (and categorical covariates) when computing per-row missingness
EXCLUDE_METADATA_COLS = [
    "cat",
    "dog",
    "multivitamin",
    "other_supplement_frequency",
    "cosmetics_frequency",
    "fermented_plant_frequency",
    "homecooked_meals_frequency",
    "meat_eggs_frequency",
    "sugary_sweets_frequency",
    "vivid_dreams",
    "sugar_sweetened_drink_frequency",
    "artificial_sweeteners",
    "olive_oil",
    "prepared_meals_frequency",
    "ready_to_eat_meals_frequency",
]
skip_cols = list(EXCLUDE_METADATA_COLS) + list(cov_cat)   # ignore these when computing per-row missingness
max_missing_frac = 0.20 # keep rows with <=10% missing across cols_check
required_for_regression = [cognition_col] + cov_num

# HELPERS
def filter_rows_by_missingness(df, *, skip_cols=(), required_cols=(), max_missing_frac=0.10):
    skip_cols = list(skip_cols)
    required_cols = list(required_cols)

    missing_skip = [c for c in skip_cols if c not in df.columns]
    if missing_skip:
        print(f"[WARN] skip_cols not in dataframe (ignored): {missing_skip[:10]}{'...' if len(missing_skip)>10 else ''}")

    cols_check = [c for c in df.columns if c not in set(skip_cols)]
    if len(cols_check) == 0:
        raise ValueError("After applying skip_cols, cols_check is empty. Reduce skip_cols.")

    missing_frac = df[cols_check].isna().mean(axis=1)
    n_in = df.shape[0]
    mask = missing_frac <= max_missing_frac
    df_f = df.loc[mask].copy()

    required_present = [c for c in required_cols if c in df_f.columns]
    df_f = df_f.dropna(subset=required_present)

    n_out = df_f.shape[0]
    report = {
        "n_in": n_in,
        "n_out": n_out,
        "dropped": n_in - n_out,
        "cols_check_n": len(cols_check),
        "missing_frac_summary": missing_frac.describe().to_dict()
    }
    return df_f, report

def build_covariates(df, cov_num, cov_cat):
    parts = []
    if cov_num:
        parts.append(df[cov_num].apply(pd.to_numeric, errors="coerce"))
    if cov_cat:
        parts.append(pd.get_dummies(
            df[cov_cat].astype("string").fillna("MISSING"),
            prefix=cov_cat,
            drop_first=True,
            dtype=float
        ))
    cov_df = pd.concat(parts, axis=1) if parts else pd.DataFrame(index=df.index)
    return cov_df

def fit_ols_hc3(y, X):
    return sm.OLS(y, X).fit(cov_type="HC3")

# LOAD MODELS + TRAINING METADATA
models_path = model_dir / "ridge_otu_models.pkl"
meta_path   = model_dir / "ridge_training_metadata.json"

ridge_models = joblib.load(models_path)
with open(meta_path, "r") as f:
    train_meta = json.load(f)

train_cols = train_meta["train_cols"]
saved_otus = train_meta["saved_otus"]

print(f"Loaded ridge OTU models: {len(ridge_models)}")
print(f"Training expects X columns: {len(train_cols)}")
print(f"OTUs to predict: {len(saved_otus)}")

Loaded ridge OTU models: 32
Training expects X columns: 29
OTUs to predict: 32


In [2]:
train_cols

['alcohol_consumption',
 'appendix_removed',
 'lactose',
 'alcohol_frequency',
 'exercise_frequency',
 'flossing_frequency',
 'frozen_dessert_frequency',
 'fruit_frequency',
 'high_fat_red_meat_frequency',
 'milk_cheese_frequency',
 'milk_substitute_frequency',
 'one_liter_of_water_a_day_frequency',
 'poultry_frequency',
 'probiotic_frequency',
 'red_meat_frequency',
 'salted_snacks_frequency',
 'seafood_frequency',
 'smoking_frequency',
 'teethbrushing_frequency',
 'vegetable_frequency',
 'vitamin_b_supplement_frequency',
 'vitamin_d_supplement_frequency',
 'whole_eggs',
 'whole_grain_frequency',
 'bowel_movement_frequency',
 'sleep_duration',
 'sex',
 'bmi',
 'weight_kg']

In [3]:
ukb_df_raw = pd.read_csv(ukb_metadata_path, index_col=ukb_index_col)
print(f"Loaded UKB-like metadata: {ukb_df_raw.shape[0]} rows, {ukb_df_raw.shape[1]} cols")

# ============================================================
# PRE-FILTER UKB QUICK DISTRIBUTION REPORT
# ============================================================

out_dir = Path("./hongrui_result/Ridge_CV_Results/UKB_PreFilter_Distribution_Check/")
out_dir.mkdir(parents=True, exist_ok=True)
out_pdf_path = out_dir / "UKB_pre_filter_distributions.pdf"

# Hard-coded: keep only these as categorical
categorical_keep = {"country_of_birth", "country", "race"}

# Plot settings
top_k_categories = 25
numeric_bins = 30
treat_numeric_as_categorical_if_unique_leq = 15

def force_numeric_everything_else(df: pd.DataFrame, categorical_keep: set) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        if c in categorical_keep:
            df[c] = df[c].astype("string")
        else:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def plot_categorical_with_missing_right(ax, s_raw: pd.Series, title: str, top_k: int = 25):
    """
    Barplot of proportions for categorical-like series.
    Ensures 'MISSING' is always the last bar (rightmost).
    Collapses tail into 'Other' (excluding MISSING).
    """
    s = s_raw.astype("string").fillna("MISSING")

    vc = s.value_counts(dropna=False)
    n = int(vc.sum()) if vc.sum() > 0 else 0

    missing_ct = int(vc.get("MISSING", 0))
    vc_no_missing = vc.drop(index=["MISSING"], errors="ignore")

    # Top-K (non-missing)
    if vc_no_missing.shape[0] > top_k:
        top = vc_no_missing.head(top_k)
        other_ct = int(vc_no_missing.iloc[top_k:].sum())
        vc_plot = top.copy()
        if other_ct > 0:
            vc_plot.loc["Other"] = other_ct
    else:
        vc_plot = vc_no_missing.copy()

    # Append MISSING as rightmost
    if missing_ct > 0:
        vc_plot.loc["MISSING"] = missing_ct

    probs = (vc_plot / max(1, n)).astype(float)

    ax.bar(probs.index.astype(str), probs.values)
    ax.set_title(f"{title} (n={n}, missing={missing_ct}={missing_ct/max(1,n):.1%})")
    ax.set_ylabel("Proportion")
    ax.tick_params(axis="x", rotation=90)

def plot_numeric_discrete_with_missing_right(ax, s_num: pd.Series, title: str, top_k: int = 25):
    """
    Discrete numeric values plotted as categorical bars (proportions).
    Ensures 'MISSING' rightmost.
    """
    n_total = int(s_num.shape[0])
    n_missing = int(s_num.isna().sum())
    s_non = s_num.dropna()

    if s_non.empty:
        # Only missing
        ax.bar(["MISSING"], [1.0])
        ax.set_title(f"{title} (n={n_total}, missing={n_missing}={n_missing/max(1,n_total):.1%})")
        ax.set_ylabel("Proportion")
        return

    vc = s_non.value_counts()
    # if too many unique, keep top_k by freq, rest -> Other
    if vc.shape[0] > top_k:
        top = vc.head(top_k)
        other_ct = int(vc.iloc[top_k:].sum())
        vc_plot = top.copy()
        vc_plot.loc["Other"] = other_ct
    else:
        vc_plot = vc.copy()

    # sort numeric keys (keep Other last-ish, and Missing last)
    def fmt_num(x):
        try:
            fx = float(x)
            if np.isfinite(fx) and fx.is_integer():
                return str(int(fx))
            return f"{fx:g}"
        except Exception:
            return str(x)

    idx = list(vc_plot.index)

    # separate numeric vs Other
    numeric_keys = [k for k in idx if k != "Other"]
    numeric_keys_sorted = sorted(numeric_keys, key=lambda k: float(k))
    ordered = numeric_keys_sorted + (["Other"] if "Other" in idx else []) + (["MISSING"] if n_missing > 0 else [])

    # build counts aligned
    counts = []
    for k in ordered:
        if k == "MISSING":
            counts.append(n_missing)
        elif k == "Other":
            counts.append(int(vc_plot.loc["Other"]))
        else:
            counts.append(int(vc_plot.loc[k]))

    probs = np.array(counts, dtype=float) / max(1, n_total)

    labels = [fmt_num(k) if k not in {"Other","MISSING"} else k for k in ordered]

    ax.bar(labels, probs)
    ax.set_title(f"{title} (n={n_total}, missing={n_missing}={n_missing/max(1,n_total):.1%})")
    ax.set_ylabel("Proportion")
    ax.tick_params(axis="x", rotation=90)

# --- Use a numeric-coerced copy ONLY for visualization (pre-filter) ---
ukb_viz = force_numeric_everything_else(ukb_df_raw, categorical_keep)

with PdfPages(out_pdf_path) as pdf:
    for col in ukb_viz.columns:
        s = ukb_viz[col]

        fig, ax = plt.subplots(1, 1, figsize=(10, 4), constrained_layout=True)

        if col in categorical_keep:
            plot_categorical_with_missing_right(ax, s, title=f"UKB pre-filter: {col}", top_k=top_k_categories)

        else:
            # numeric column (after coercion)
            nunq = int(s.dropna().nunique())

            if nunq <= treat_numeric_as_categorical_if_unique_leq:
                plot_numeric_discrete_with_missing_right(ax, s, title=f"UKB pre-filter: {col}", top_k=top_k_categories)
            else:
                # continuous numeric histogram (missing shown in title)
                x = s.dropna().values.astype(float)
                n_total = int(len(s))
                n_missing = int(s.isna().sum())
                if len(x) == 0:
                    ax.text(0.5, 0.5, "No non-missing numeric values", ha="center", va="center")
                    ax.set_axis_off()
                else:
                    bins = np.histogram_bin_edges(x, bins=numeric_bins)
                    ax.hist(x, bins=bins)
                    ax.set_ylabel("Count")
                ax.set_title(f"UKB pre-filter: {col} (n={n_total}, missing={n_missing}={n_missing/max(1,n_total):.1%})")

        pdf.savefig(fig)
        plt.close(fig)

print(f"Saved pre-filter UKB distribution PDF to:\n  {out_pdf_path}")

Loaded UKB-like metadata: 502244 rows, 48 cols
Saved pre-filter UKB distribution PDF to:
  hongrui_result/Ridge_CV_Results/UKB_PreFilter_Distribution_Check/UKB_pre_filter_distributions.pdf


In [4]:
# ============================================================
# Reduce UKB dataframe to ONLY the columns needed downstream
# (prevents row-missingness filtering from being dominated by unrelated/high-missing columns)
# ============================================================
needed_cols = sorted(set(train_cols) | set([cognition_col] + cov_num + cov_cat))
present_needed = [c for c in needed_cols if c in ukb_df_raw.columns]
missing_needed = [c for c in needed_cols if c not in ukb_df_raw.columns]
if missing_needed:
    print(f"[WARN] UKB is missing {len(missing_needed)} / {len(needed_cols)} needed cols (they will be treated as all-NA at prediction time).")
    print("First 25 missing needed cols:", missing_needed[:25])

ukb_df = ukb_df_raw[present_needed].copy()

# Optional row filtering (keeps analysis sample sane)
ukb_df, miss_report = filter_rows_by_missingness(
    ukb_df,
    skip_cols=skip_cols,
    required_cols=required_for_regression,
    max_missing_frac=max_missing_frac
)
print("\n[ROW FILTER REPORT]")
print(f"Input rows:   {miss_report['n_in']}")
print(f"Output rows:  {miss_report['n_out']}")
print(f"Dropped rows: {miss_report['dropped']}")
print(f"Cols counted toward missingness: {miss_report['cols_check_n']}")

[WARN] skip_cols not in dataframe (ignored): ['cat', 'dog', 'multivitamin', 'other_supplement_frequency', 'cosmetics_frequency', 'fermented_plant_frequency', 'homecooked_meals_frequency', 'meat_eggs_frequency', 'sugary_sweets_frequency', 'vivid_dreams']...

[ROW FILTER REPORT]
Input rows:   502244
Output rows:  97074
Dropped rows: 405170
Cols counted toward missingness: 31


In [5]:
# CONFIG
# -----------------------------
agp_metadata_path = "./Data/Cleaned_data/processed_metadata.csv"
agp_index_col = "sample_name"

out_dir = Path("./hongrui_result/Ridge_CV_Results/UKB_AGP_Distribution_Check/")
out_dir.mkdir(parents=True, exist_ok=True)

out_pdf_path = out_dir / "AGP_vs_UKB_distributions.pdf"
out_summary_csv = out_dir / "AGP_vs_UKB_drift_summary.csv"

cols_to_compare = None  # or e.g. train_cols if you want only predictors used by ridge

top_k_categories = 20
numeric_bins = 30
treat_numeric_as_categorical_if_unique_leq = 15

# Only these remain categorical
categorical_keep = {"country_of_birth", "country", "race"}

# -----------------------------
# HELPERS (minimal)
# -----------------------------
def _as_categorical(series: pd.Series) -> pd.Series:
    return series.astype("string").fillna("MISSING")

def _tv_distance(p: pd.Series, q: pd.Series) -> float:
    return float(0.5 * np.abs(p.values - q.values).sum())

def _std_mean_diff(x: np.ndarray, y: np.ndarray) -> float:
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    if len(x) < 2 or len(y) < 2:
        return np.nan
    mx, my = np.mean(x), np.mean(y)
    sx, sy = np.std(x, ddof=1), np.std(y, ddof=1)
    sp = np.sqrt((sx**2 + sy**2) / 2.0)
    return float((my - mx) / sp) if sp > 0 else np.nan

def _plot_numeric(ax, vals, bins, title):
    ax.hist(vals, bins=bins)
    ax.set_title(title)
    ax.set_ylabel("Count")

def _plot_categorical(ax, probs: pd.Series, title):
    ax.bar(probs.index.astype(str), probs.values)
    ax.set_title(title)
    ax.set_ylabel("Proportion")
    ax.tick_params(axis="x", rotation=90)

def force_numeric_everything_else(df: pd.DataFrame, categorical_keep: set) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        if c in categorical_keep:
            df[c] = df[c].astype("string")
        else:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def clean_num_tick(x):
    # 1.0 -> "1", 1.25 -> "1.25"
    try:
        fx = float(x)
        if np.isfinite(fx) and fx.is_integer():
            return str(int(fx))
        return f"{fx:g}"
    except Exception:
        return str(x)

# -----------------------------
# LOAD AGP + use FILTERED ukb_df (already in memory)
# -----------------------------
agp_df = pd.read_csv(agp_metadata_path, index_col=agp_index_col)

print(f"AGP metadata loaded: {agp_df.shape}")
print(f"UKB (FILTERED) loaded: {ukb_df.shape}  <-- using your filtered dataframe")

# Force numeric typing (prevents '1' vs '1.0')
agp_df = force_numeric_everything_else(agp_df, categorical_keep)
ukb_df_num = force_numeric_everything_else(ukb_df, categorical_keep)  # don't overwrite if you want

# Shared columns (post-typing)
shared_cols = sorted(set(agp_df.columns).intersection(set(ukb_df_num.columns)))
print(f"Shared columns between AGP and filtered UKB: {len(shared_cols)}")

if cols_to_compare is not None:
    shared_cols = [c for c in shared_cols if c in set(cols_to_compare)]
    print(f"After cols_to_compare restriction: {len(shared_cols)}")

# -----------------------------
# ITERATE + SAVE MULTIPAGE PDF
# -----------------------------
drift_rows = []

with PdfPages(out_pdf_path) as pdf:
    for col in shared_cols:
        s_agp = agp_df[col]
        s_ukb = ukb_df_num[col]

        fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

        if col in categorical_keep:
            c_agp = _as_categorical(s_agp)
            c_ukb = _as_categorical(s_ukb)

            vc_agp = c_agp.value_counts(normalize=True, dropna=False)
            vc_ukb = c_ukb.value_counts(normalize=True, dropna=False)

            top_union = list(pd.Index(vc_agp.head(top_k_categories).index).union(vc_ukb.head(top_k_categories).index))
            p = vc_agp.reindex(top_union, fill_value=0.0)
            q = vc_ukb.reindex(top_union, fill_value=0.0)

            avg = (p + q) / 2.0
            keep = avg.sort_values(ascending=False).head(top_k_categories).index
            p2 = p.reindex(keep).copy()
            q2 = q.reindex(keep).copy()

            p2.loc["Other"] = float(1.0 - p2.sum())
            q2.loc["Other"] = float(1.0 - q2.sum())

            _plot_categorical(axes[0], p2, f"AGP: {col} (n={c_agp.notna().sum()})")
            _plot_categorical(axes[1], q2, f"UKB: {col} (n={c_ukb.notna().sum()})")

            tv = _tv_distance(p2 / p2.sum(), q2 / q2.sum())

            drift_rows.append({
                "column": col,
                "type": "categorical",
                "agp_n_nonmissing": int(s_agp.notna().sum()),
                "ukb_n_nonmissing": int(s_ukb.notna().sum()),
                "tv_distance": tv,
                "std_mean_diff": np.nan
            })

        else:
            nunq = int(max(s_agp.dropna().nunique(), s_ukb.dropna().nunique()))

            if nunq <= treat_numeric_as_categorical_if_unique_leq:
                vc_agp = s_agp.dropna().value_counts(normalize=True)
                vc_ukb = s_ukb.dropna().value_counts(normalize=True)

                cats = sorted(set(vc_agp.index).union(set(vc_ukb.index)))
                p2 = vc_agp.reindex(cats, fill_value=0.0)
                q2 = vc_ukb.reindex(cats, fill_value=0.0)

                if len(cats) > top_k_categories:
                    avg = (p2 + q2) / 2.0
                    keep = avg.sort_values(ascending=False).head(top_k_categories).index
                    p2 = p2.reindex(keep).copy()
                    q2 = q2.reindex(keep).copy()
                    p2.loc["Other"] = float(1.0 - p2.sum())
                    q2.loc["Other"] = float(1.0 - q2.sum())

                # clean tick labels
                labels = [clean_num_tick(x) if x != "Other" else "Other" for x in p2.index]
                p2.index = labels
                q2.index = labels

                _plot_categorical(axes[0], p2, f"AGP: {col} (n={int(s_agp.notna().sum())})")
                _plot_categorical(axes[1], q2, f"UKB: {col} (n={int(s_ukb.notna().sum())})")

                tv = _tv_distance(p2 / p2.sum(), q2 / q2.sum()) if (p2.sum() > 0 and q2.sum() > 0) else np.nan

                drift_rows.append({
                    "column": col,
                    "type": "numeric_discrete",
                    "agp_n_nonmissing": int(s_agp.notna().sum()),
                    "ukb_n_nonmissing": int(s_ukb.notna().sum()),
                    "tv_distance": tv,
                    "std_mean_diff": np.nan
                })

            else:
                x = s_agp.dropna().values.astype(float)
                y = s_ukb.dropna().values.astype(float)

                if len(x) == 0 or len(y) == 0:
                    axes[0].text(0.5, 0.5, f"No numeric data (AGP)\n{col}", ha="center", va="center")
                    axes[1].text(0.5, 0.5, f"No numeric data (UKB)\n{col}", ha="center", va="center")
                    axes[0].set_axis_off()
                    axes[1].set_axis_off()
                    smd = np.nan
                else:
                    combined = np.concatenate([x, y])
                    bins = np.histogram_bin_edges(combined, bins=numeric_bins)

                    _plot_numeric(axes[0], x, bins, f"AGP: {col} (n={len(x)})")
                    _plot_numeric(axes[1], y, bins, f"UKB: {col} (n={len(y)})")

                    lo, hi = float(np.nanmin(combined)), float(np.nanmax(combined))
                    axes[0].set_xlim(lo, hi)
                    axes[1].set_xlim(lo, hi)

                    smd = _std_mean_diff(x, y)

                drift_rows.append({
                    "column": col,
                    "type": "numeric_continuous",
                    "agp_n_nonmissing": int(len(x)),
                    "ukb_n_nonmissing": int(len(y)),
                    "tv_distance": np.nan,
                    "std_mean_diff": smd
                })

        fig.suptitle(f"Distribution check: {col}", y=1.02, fontsize=12)
        pdf.savefig(fig)
        plt.close(fig)

# Save drift summary
drift_df = pd.DataFrame(drift_rows)
drift_df["shift_score"] = np.where(
    drift_df["tv_distance"].notna(),
    drift_df["tv_distance"].astype(float),
    np.abs(drift_df["std_mean_diff"].astype(float))
)
drift_df = drift_df.sort_values("shift_score", ascending=False)
drift_df.to_csv(out_summary_csv, index=False)

print(f"\nSaved PDF to: {out_pdf_path}")
print(f"Saved drift CSV to: {out_summary_csv}")
print("\nTop 20 most-shifted columns:")
print(drift_df[["column","type","shift_score","tv_distance","std_mean_diff","agp_n_nonmissing","ukb_n_nonmissing"]].head(20).to_string(index=False))

AGP metadata loaded: (9559, 48)
UKB (FILTERED) loaded: (97074, 33)  <-- using your filtered dataframe
Shared columns between AGP and filtered UKB: 32

Saved PDF to: hongrui_result/Ridge_CV_Results/UKB_AGP_Distribution_Check/AGP_vs_UKB_distributions.pdf
Saved drift CSV to: hongrui_result/Ridge_CV_Results/UKB_AGP_Distribution_Check/AGP_vs_UKB_drift_summary.csv

Top 20 most-shifted columns:
                            column               type  shift_score  tv_distance  std_mean_diff  agp_n_nonmissing  ukb_n_nonmissing
                    sleep_duration   numeric_discrete     0.987342     0.987342            NaN              9559             96889
               vegetable_frequency   numeric_discrete     0.843389     0.843389            NaN              9559             95460
                     age_corrected numeric_continuous     0.767111          NaN       0.767111              9215             97074
                  country_of_birth        categorical     0.748735     0.748735      

In [6]:
# BUILD UKB X (exact training cols)
X_ukb = ukb_df.reindex(columns=train_cols)
X_ukb = X_ukb.apply(pd.to_numeric, errors="coerce")

missing_cols = [c for c in train_cols if c not in ukb_df.columns]
print(f"UKB is missing {len(missing_cols)} / {len(train_cols)} training columns.")
if missing_cols:
    print("First 25 missing cols:", missing_cols[:25])

missing_rate = float(np.mean(pd.isna(X_ukb.values)))
print(f"Overall missing rate in X_ukb (after numeric coercion): {missing_rate*100:.2f}%")

# PREDICT OTU PROXIES IN UKB
otu_hat = pd.DataFrame(index=ukb_df.index)

for otu in saved_otus:
    if otu not in ridge_models:
        print(f"[WARN] OTU model missing in ridge_models.pkl: {otu} (skipping)")
        continue
    otu_hat[f"{otu}__hat"] = ridge_models[otu].predict(X_ukb)

otu_hat_path = out_dir / "ukb_predicted_otus.csv"
otu_hat.to_csv(otu_hat_path)
print(f"Saved predicted OTU proxies to: {otu_hat_path}")

# ============================================================
# REGRESSION: fluid intelligence ~ OTU_hat + covariates (UNIVARIATE per OTU)
# ============================================================
if cognition_col not in ukb_df.columns:
    raise ValueError(f"Outcome column not found: {cognition_col}")

y = pd.to_numeric(ukb_df[cognition_col], errors="coerce")
cov_df = build_covariates(ukb_df, cov_num=cov_num, cov_cat=cov_cat)

# Lock sample using outcome + covariates (predictions have no NA because pipeline imputes)
base = pd.concat([y.rename("y"), cov_df], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
print(f"\nRows available for regression after locking y+covariates: {base.shape[0]}")

y_clean = base["y"].astype(float)
cov_clean = base.drop(columns=["y"]).astype(float)
cov_clean = sm.add_constant(cov_clean, has_constant="add")

reg_rows = []

for col in otu_hat.columns:
    # align with locked sample
    x1 = otu_hat.loc[base.index, col].astype(float)
    Xmat = pd.concat([cov_clean, x1.rename(col)], axis=1)

    fit = fit_ols_hc3(y_clean, Xmat)

    reg_rows.append({
        "otu_hat": col,
        "beta": float(fit.params[col]),
        "se_hc3": float(fit.bse[col]),
        "t_hc3": float(fit.tvalues[col]),
        "p_value": float(fit.pvalues[col]),
        "n": int(fit.nobs),
        "r2": float(fit.rsquared),
        "adj_r2": float(fit.rsquared_adj),
    })

reg_df = pd.DataFrame(reg_rows).sort_values("p_value")

# BH-FDR across OTUs
if reg_df.shape[0] > 0:
    rej, qvals, _, _ = multipletests(reg_df["p_value"].values, alpha=0.05, method="fdr_bh")
    reg_df["q_value_BH"] = qvals
    reg_df["reject_FDR_0p05"] = rej

reg_path = out_dir / "ukb_fluid_intelligence_on_otu_hat_univariate.csv"
reg_df.to_csv(reg_path, index=False)
print(f"Saved univariate regression results to: {reg_path}")

print("\nTop 20 OTU proxies by p-value:")
print(reg_df.head(20).to_string(index=False))

# ============================================================
# OPTIONAL: one JOINT model with ALL OTU_hats + covariates
# (can be unstable if many OTUs; use only if dimensionality is reasonable)
# ============================================================
do_joint = False
if do_joint:
    X_joint = pd.concat([cov_df, otu_hat], axis=1)
    X_joint = sm.add_constant(X_joint, has_constant="add").replace([np.inf, -np.inf], np.nan)
    joint_data = pd.concat([y.rename("y"), X_joint], axis=1).dropna()
    yj = joint_data["y"].astype(float)
    Xj = joint_data.drop(columns=["y"]).astype(float)

    fit_joint = fit_ols_hc3(yj, Xj)
    joint_path = out_dir / "ukb_joint_model_summary.txt"
    with open(joint_path, "w") as f:
        f.write(fit_joint.summary().as_text())
    print(f"Saved JOINT model summary to: {joint_path}")

UKB is missing 0 / 29 training columns.
Overall missing rate in X_ukb (after numeric coercion): 14.43%
Saved predicted OTU proxies to: hongrui_result/Ridge_CV_Results/UKB_AGP_Distribution_Check/ukb_predicted_otus.csv

Rows available for regression after locking y+covariates: 97074
Saved univariate regression results to: hongrui_result/Ridge_CV_Results/UKB_AGP_Distribution_Check/ukb_fluid_intelligence_on_otu_hat_univariate.csv

Top 20 OTU proxies by p-value:
                                                         otu_hat      beta   se_hc3      t_hc3       p_value     n       r2   adj_r2    q_value_BH  reject_FDR_0p05
                   Ruminococcaceae_Intestinimonas__1177c4dc__hat  0.803464 0.023951  33.546668 1.006872e-246 97074 0.058071 0.057993 3.221992e-245             True
                        Lachnospiraceae_Roseburia__daf013c0__hat  0.402214 0.013292  30.259373 3.927787e-201 97074 0.055959 0.055881 6.284460e-200             True
                  Ruminococcaceae_Subdoligranu